# 01 -- Perception: Keystroke Dynamics

**Author**: Tamer Atesyakar

This notebook walks through the 32-dimensional `InteractionFeatureVector` produced by the `i3.interaction.features` module. We record a short typing sample, compute the four feature groups (temporal, behavioural, content, contextual), and visualise each group with matplotlib.

**Citations**
- Banerjee & Woodard (2012). *Biometric Authentication and Identification using Keystroke Dynamics.*
- Epp, Lippold & Mandryk (2011). *Identifying Emotional States using Keystroke Dynamics.* CHI.
- Picard (1997). *Affective Computing.* MIT Press.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import asdict

# Palette
C_COLD = '#0f3460'
C_HOT  = '#e94560'

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.25,
    'font.size': 10,
})

## 1.1 A synthetic typing sample

We simulate 120 keystrokes with the structure of a real message: quick bursts of in-word typing, longer pauses at word boundaries, and occasional backspaces.

In [ ]:
rng = np.random.default_rng(42)

def synth_sample(n=120):
    keys, times, dwell, kinds = [], [], [], []
    t = 0.0
    for i in range(n):
        # Word-internal vs boundary pause
        iki = rng.lognormal(mean=4.6, sigma=0.35)
        if i > 0 and i % 7 == 0:
            iki *= 2.4  # pause at word boundary
        t += iki
        times.append(t)
        dwell.append(max(20.0, rng.normal(70, 12)))
        is_bs = rng.random() < 0.07
        kinds.append('backspace' if is_bs else 'char')
        keys.append('\x08' if is_bs else chr(ord('a') + rng.integers(0, 26)))
    return np.array(times), np.array(dwell), kinds, keys

times, dwell, kinds, keys = synth_sample(120)
print(f'n = {len(times)}, duration = {times[-1]/1000:.2f} s')

## 1.2 The four feature groups

The `InteractionFeatureVector` is organised into:
1. **Temporal** -- mean/std inter-key interval, pause counts, rhythm consistency.
2. **Behavioural** -- backspace rate, edits per second, composition time.
3. **Content** -- message length, lexical diversity, punctuation ratios.
4. **Contextual** -- time of day, session position, recent engagement.

We compute each group from the raw keystroke stream.

In [ ]:
iki = np.diff(times)  # inter-key intervals in ms
n_chars = sum(1 for k in kinds if k == 'char')
n_bs    = sum(1 for k in kinds if k == 'backspace')

# Temporal group (8 dims)
temporal = {
    'mean_iki':   float(np.mean(iki)),
    'std_iki':    float(np.std(iki)),
    'median_iki': float(np.median(iki)),
    'q90_iki':    float(np.quantile(iki, 0.9)),
    'min_iki':    float(np.min(iki)),
    'max_iki':    float(np.max(iki)),
    'rhythm_cv':  float(np.std(iki) / (np.mean(iki) + 1e-6)),
    'pause_ct':   float(np.sum(iki > 500)),
}
temporal

In [ ]:
# Behavioural group (8 dims)
duration_s = (times[-1] - times[0]) / 1000.0
behavioural = {
    'backspace_rate':  n_bs / max(n_chars, 1),
    'edits_per_sec':   n_bs / max(duration_s, 1e-3),
    'composition_s':   duration_s,
    'mean_dwell':      float(np.mean(dwell)),
    'std_dwell':       float(np.std(dwell)),
    'chars_per_sec':   n_chars / max(duration_s, 1e-3),
    'keystroke_total': float(len(times)),
    'error_bursts':    float(sum(1 for i in range(1, len(kinds))
                                 if kinds[i]=='backspace' and kinds[i-1]=='backspace')),
}
behavioural

In [ ]:
# Content group (8 dims) -- from the reconstructed text
text_chars = [k for k, kd in zip(keys, kinds) if kd == 'char']
text = ''.join(text_chars)
content = {
    'len':          float(len(text)),
    'unique_ratio': len(set(text)) / max(len(text), 1),
    'vowel_ratio':  sum(c in 'aeiou' for c in text) / max(len(text), 1),
    'caps_ratio':   0.0,  # lowercase synth
    'punct_ratio':  0.0,
    'digit_ratio':  0.0,
    'space_ratio':  0.0,
    'repeat_ratio': sum(text[i] == text[i-1] for i in range(1, len(text))) / max(len(text)-1, 1),
}
content

In [ ]:
# Contextual group (8 dims)
import time as _time
now = _time.localtime()
contextual = {
    'hour_norm':      now.tm_hour / 23.0,
    'weekday_norm':   now.tm_wday / 6.0,
    'session_pos':    0.25,    # 1st message of session
    'recent_engage':  0.60,    # placeholder EMA
    'msg_since_rest': 0.0,
    'device_kind':    0.0,     # 0 = desktop
    'input_method':   0.0,     # 0 = physical kb
    'lang_signal':    0.0,     # 0 = en
}
contextual

## 1.3 Assemble the 32-dim InteractionFeatureVector

In production, `i3.interaction.features.compute_feature_vector` handles this. Here we concatenate the four groups and verify the resulting shape.

In [ ]:
try:
    from i3.interaction import features as _feat  # noqa: F401
    print('i3.interaction.features imported ok')
except Exception as e:
    print(f'(i3.interaction.features not importable in this kernel: {e})')

vec = np.array(
    list(temporal.values())
    + list(behavioural.values())
    + list(content.values())
    + list(contextual.values()),
    dtype=np.float32,
)
print('shape:', vec.shape)
print('dtype:', vec.dtype)

## 1.4 Visualising the four groups

In [ ]:
labels = ['Temporal', 'Behavioural', 'Content', 'Contextual']
groups = [temporal, behavioural, content, contextual]
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, lab, g in zip(axes.flat, labels, groups):
    keys_ = list(g.keys())
    vals  = [float(v) for v in g.values()]
    # Normalise each group for display only.
    m = max(abs(v) for v in vals) or 1.0
    norm = [v / m for v in vals]
    ax.barh(keys_, norm, color=[C_HOT if v >= 0 else C_COLD for v in norm])
    ax.set_title(lab)
    ax.axvline(0, color='#333', lw=0.5)
plt.tight_layout()
plt.show()

## 1.5 Inter-key interval histogram

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(iki, bins=30, color=C_COLD, edgecolor='white')
ax.axvline(np.mean(iki), color=C_HOT, lw=2, label=f'mean={np.mean(iki):.0f} ms')
ax.axvline(np.median(iki), color='#333', lw=1.2, ls='--', label=f'median={np.median(iki):.0f} ms')
ax.set_xlabel('Inter-key interval (ms)')
ax.set_ylabel('count')
ax.set_title('IKI distribution -- log-normal shape is typical')
ax.legend()
plt.tight_layout()
plt.show()

## 1.6 Rhythm trace over time

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(times[1:] / 1000.0, iki, color=C_COLD, lw=1.0, alpha=0.85)
ax.scatter(times[1:][iki > 500] / 1000.0, iki[iki > 500],
           color=C_HOT, s=20, zorder=3, label='pause > 500 ms')
ax.set_xlabel('time (s)')
ax.set_ylabel('IKI (ms)')
ax.set_title('Rhythm trace -- pauses cluster at word boundaries')
ax.legend()
plt.tight_layout()
plt.show()

## 1.7 Takeaways

- The four feature groups are linearly independent: temporal captures *when*, behavioural captures *how fluently*, content captures *what*, contextual captures *the surrounding conditions*.
- The log-normal IKI distribution is a hallmark of natural typing; deviations toward heavier tails are a strong signal of cognitive load (Epp et al. 2011).
- The 32-dim vector is the input to the TCN encoder covered in notebook 02.